In [3]:
# ============================================================
# KNN CLASSIFICATION
# DATASET UPLOAD + FEATURE SELECTION + TARGET SELECTION
# + K SELECTION + RESULTS + ANIMATION
# GOOGLE COLAB VERSION
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import io
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import ipywidgets as widgets


# Maximum size of embedded animation = 100 MB
mpl.rcParams["animation.embed_limit"] = 100


# ============================================================
# 2. TITLE
# ============================================================

print("=" * 60)
print("                  KNN CLASSIFICATION")
print("=" * 60)

print("\nUpload your dataset.")
print("Supported formats: CSV, XLSX, XLS")


# ============================================================
# 3. UPLOAD DATASET
# ============================================================

try:

    from google.colab import files

    uploaded = files.upload()

    if len(uploaded) == 0:
        raise ValueError("No file was uploaded.")

    filename = list(uploaded.keys())[0]

except ImportError:

    raise RuntimeError(
        "This program is designed for Google Colab. "
        "Please run it in Google Colab."
    )


print("\nUploaded file:", filename)


# ============================================================
# 4. READ DATASET
# ============================================================

try:

    # --------------------------------------------------------
    # CSV
    # --------------------------------------------------------

    if filename.lower().endswith(".csv"):

        try:

            df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                sep=None,
                engine="python"
            )

        except UnicodeDecodeError:

            df = pd.read_csv(
                io.BytesIO(uploaded[filename]),
                sep=None,
                engine="python",
                encoding="latin1"
            )


    # --------------------------------------------------------
    # EXCEL
    # --------------------------------------------------------

    elif filename.lower().endswith((".xlsx", ".xls")):

        df = pd.read_excel(
            io.BytesIO(uploaded[filename])
        )


    # --------------------------------------------------------
    # UNSUPPORTED FILE
    # --------------------------------------------------------

    else:

        raise ValueError(
            "Unsupported file format. "
            "Please upload CSV, XLSX or XLS."
        )


except Exception as e:

    print("\nERROR READING DATASET")
    print("-" * 60)
    print(e)

    raise


# ============================================================
# 5. CLEAN DATASET
# ============================================================

# Remove completely empty rows

df = df.dropna(
    axis=0,
    how="all"
)


# Remove completely empty columns

df = df.dropna(
    axis=1,
    how="all"
)


# Clean column names

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)


# ============================================================
# 6. CHECK DATASET
# ============================================================

if df.empty:

    raise ValueError(
        "The uploaded dataset is empty."
    )


if len(df.columns) < 3:

    raise ValueError(
        "The dataset must contain at least "
        "3 columns: two features and one target."
    )


# ============================================================
# 7. DISPLAY DATASET INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print("\nNumber of Rows    :", df.shape[0])
print("Number of Columns :", df.shape[1])


print("\nColumn Names:")

for i, column in enumerate(df.columns):

    print(
        f"{i + 1}. {column}"
    )


print("\nFirst 5 Rows:")

display(df.head())


# ============================================================
# 8. DETERMINE NUMERICAL COLUMNS
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()


print("\n" + "=" * 60)
print("NUMERICAL FEATURES")
print("=" * 60)


if len(numeric_columns) < 2:

    raise ValueError(
        "At least TWO numerical feature columns "
        "are required for the 2-D KNN visualization."
    )


for column in numeric_columns:

    print("•", column)


# ============================================================
# 9. FEATURE SELECTION
# ============================================================

print("\n" + "=" * 60)
print("SELECT TWO FEATURES")
print("=" * 60)

print(
    "\nYour dataset contains",
    len(numeric_columns),
    "numerical features."
)

print(
    "Select exactly TWO different numerical features "
    "for the KNN visualization."
)


# ------------------------------------------------------------
# X-AXIS FEATURE
# ------------------------------------------------------------

x_feature_dropdown = widgets.Dropdown(

    options=numeric_columns,

    value=numeric_columns[0],

    description="X-axis:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="400px"
    )
)


# ------------------------------------------------------------
# Y-AXIS FEATURE
# ------------------------------------------------------------

y_feature_dropdown = widgets.Dropdown(

    options=numeric_columns,

    value=numeric_columns[1],

    description="Y-axis:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="400px"
    )
)


display(x_feature_dropdown)
display(y_feature_dropdown)


# ============================================================
# 10. TARGET COLUMN SELECTION
# ============================================================

print("\n" + "=" * 60)
print("SELECT TARGET COLUMN")
print("=" * 60)


# Automatically select Purchased if it exists.
# Otherwise select the last column.

if "Purchased" in df.columns:

    default_target = "Purchased"

else:

    default_target = df.columns[-1]


target_dropdown = widgets.Dropdown(

    options=list(df.columns),

    value=default_target,

    description="Target:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="400px"
    )
)


display(target_dropdown)


# ============================================================
# 11. K VALUE SELECTION
# ============================================================

print("\n" + "=" * 60)
print("SELECT VALUE OF K")
print("=" * 60)


k_slider = widgets.IntSlider(

    value=3,

    min=1,

    max=15,

    step=1,

    description="K:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="400px"
    )
)


display(k_slider)


# ============================================================
# 12. RUN BUTTON
# ============================================================

run_button = widgets.Button(

    description="Run KNN",

    button_style="success",

    icon="play",

    layout=widgets.Layout(
        width="200px",
        height="40px"
    )
)


display(run_button)


# ============================================================
# 13. OUTPUT AREA
# ============================================================

output_area = widgets.Output()

display(output_area)


# ============================================================
# 14. KNN FUNCTION
# ============================================================

def run_knn(button):

    # --------------------------------------------------------
    # Clear previous output
    # --------------------------------------------------------

    with output_area:

        output_area.clear_output(wait=True)


        # ====================================================
        # GET USER SELECTIONS
        # ====================================================

        x_feature = x_feature_dropdown.value

        y_feature = y_feature_dropdown.value

        target_column = target_dropdown.value

        K = k_slider.value


        print("\n")
        print("=" * 60)
        print("                  KNN ANALYSIS")
        print("=" * 60)


        print("\nX-axis Feature :", x_feature)

        print("Y-axis Feature :", y_feature)

        print("Target Column  :", target_column)

        print("K              :", K)


        # ====================================================
        # 15. CHECK FEATURE SELECTION
        # ====================================================

        if x_feature == y_feature:

            print("\nERROR")
            print("-" * 60)

            print(
                "X-axis and Y-axis features must be different."
            )

            print(
                "\nPlease select TWO different numerical features."
            )

            return


        # ====================================================
        # 16. DEFINE FEATURES
        # ====================================================

        feature_columns = [
            x_feature,
            y_feature
        ]


        print("\nFeatures used:")

        for feature in feature_columns:

            print("•", feature)


        # ====================================================
        # 17. CREATE MODEL DATA
        # ====================================================

        data = df[
            feature_columns + [target_column]
        ].copy()


        # ====================================================
        # 18. REMOVE MISSING VALUES
        # ====================================================

        before_rows = len(data)


        data = data.dropna()


        removed_rows = (
            before_rows -
            len(data)
        )


        if removed_rows > 0:

            print(
                "\nMissing-value rows removed:",
                removed_rows
            )


        # ====================================================
        # 19. CHECK SAMPLE SIZE
        # ====================================================

        if len(data) < 10:

            print("\nERROR")
            print("-" * 60)

            print(
                "Dataset contains too few valid "
                "samples after removing missing values."
            )

            return


        # ====================================================
        # 20. CREATE X AND Y
        # ====================================================

        X = data[
            feature_columns
        ]

        y = data[
            target_column
        ]


        # ====================================================
        # 21. ENCODE TARGET
        # ====================================================

        label_encoder = LabelEncoder()


        y_encoded = label_encoder.fit_transform(
            y.astype(str)
        )


        print("\n" + "=" * 60)
        print("CLASS INFORMATION")
        print("=" * 60)


        print("\nClasses:")


        for i, class_name in enumerate(
            label_encoder.classes_
        ):

            print(
                f"{i} → {class_name}"
            )


        # ====================================================
        # 22. CLASS DISTRIBUTION
        # ====================================================

        unique_classes, class_counts = np.unique(
            y_encoded,
            return_counts=True
        )


        print("\nClass Distribution:")


        for class_id, count in zip(
            unique_classes,
            class_counts
        ):

            print(
                f"{label_encoder.classes_[class_id]} : {count}"
            )


        # ====================================================
        # 23. CHECK NUMBER OF CLASSES
        # ====================================================

        if len(unique_classes) < 2:

            print("\nERROR")
            print("-" * 60)

            print(
                "Target column must contain "
                "at least TWO classes."
            )

            return


        # ====================================================
        # 24. TRAIN / TEST SPLIT
        # ====================================================

        try:

            X_train, X_test, y_train, y_test = (
                train_test_split(

                    X,

                    y_encoded,

                    test_size=0.20,

                    random_state=42,

                    stratify=y_encoded
                )
            )


        except ValueError:

            print(
                "\nWarning: Stratified split could not "
                "be performed."
            )

            print(
                "Using a regular random split instead."
            )


            X_train, X_test, y_train, y_test = (
                train_test_split(

                    X,

                    y_encoded,

                    test_size=0.20,

                    random_state=42
                )
            )


        # ====================================================
        # 25. FEATURE SCALING
        # ====================================================

        scaler = StandardScaler()


        X_train_scaled = scaler.fit_transform(
            X_train
        )


        X_test_scaled = scaler.transform(
            X_test
        )


        # ====================================================
        # 26. CHECK K
        # ====================================================

        if K > len(X_train_scaled):

            print("\nERROR")
            print("-" * 60)

            print(
                f"K = {K} is greater than the "
                f"number of training samples "
                f"({len(X_train_scaled)})."
            )

            print(
                f"\nPlease choose K <= "
                f"{len(X_train_scaled)}."
            )

            return


        # ====================================================
        # 27. CREATE KNN MODEL
        # ====================================================

        model = KNeighborsClassifier(

            n_neighbors=K,

            metric="euclidean"
        )


        # ====================================================
        # 28. TRAIN MODEL
        # ====================================================

        model.fit(

            X_train_scaled,

            y_train
        )


        # ====================================================
        # 29. PREDICT
        # ====================================================

        y_pred = model.predict(

            X_test_scaled
        )


        # ====================================================
        # 30. ACCURACY
        # ====================================================

        accuracy = accuracy_score(

            y_test,

            y_pred
        )


        # ====================================================
        # 31. RESULTS
        # ====================================================

        print("\n")
        print("=" * 60)
        print("                  RESULTS")
        print("=" * 60)


        print(
            "\nTotal Samples       :",
            len(data)
        )


        print(
            "Training Samples    :",
            len(X_train)
        )


        print(
            "Testing Samples     :",
            len(X_test)
        )


        print(
            "Number of Features  :",
            len(feature_columns)
        )


        print(
            "K                   :",
            K
        )


        print(
            "\nAccuracy            :",
            f"{accuracy * 100:.2f}%"
        )


        # ====================================================
        # 32. CONFUSION MATRIX
        # ====================================================

        all_labels = np.arange(
            len(label_encoder.classes_)
        )


        cm = confusion_matrix(

            y_test,

            y_pred,

            labels=all_labels
        )


        print("\nConfusion Matrix:")

        print(cm)


        # ====================================================
        # 33. CLASSIFICATION REPORT
        # ====================================================

        print("\nClassification Report:")


        print(

            classification_report(

                y_test,

                y_pred,

                labels=all_labels,

                target_names=[

                    str(x)

                    for x in label_encoder.classes_
                ],

                zero_division=0
            )
        )


        # ====================================================
        # 34. ANIMATION
        # ====================================================

        print("\n")
        print("=" * 60)
        print("           STARTING KNN ANIMATION")
        print("=" * 60)


        # ====================================================
        # 35. USE TRAINING DATA FOR VISUALIZATION
        # ====================================================

        X_visual = X_train_scaled

        X_test_visual = X_test_scaled


        # ====================================================
        # 36. SELECT FIRST TEST POINT
        # ====================================================

        test_point = X_test_visual[0]


        actual_class = y_test[0]


        predicted_class = model.predict(

            test_point.reshape(1, -1)
        )[0]


        # ====================================================
        # 37. CALCULATE EUCLIDEAN DISTANCES
        # ====================================================

        distances = np.sqrt(

            np.sum(

                (
                    X_visual -
                    test_point
                ) ** 2,

                axis=1
            )
        )


        # ====================================================
        # 38. SORT DISTANCES
        # ====================================================

        sorted_indices = np.argsort(
            distances
        )


        # ====================================================
        # 39. FIND K NEAREST NEIGHBOURS
        # ====================================================

        nearest_indices = sorted_indices[:K]


        # ====================================================
        # 40. ANIMATION ORDER
        # ====================================================
        #
        # Show points from nearest to farthest.
        # This makes the KNN concept easier to understand.
        # ====================================================

        max_distance_frames = 30


        total_training_points = len(
            X_visual
        )


        if total_training_points <= max_distance_frames:

            animation_indices = sorted_indices


        else:

            animation_indices = sorted_indices[
                :max_distance_frames
            ]


        # ====================================================
        # 41. CREATE FIGURE
        # ====================================================

        fig, ax = plt.subplots(

            figsize=(9, 6)
        )


        # ====================================================
        # 42. PLOT TRAINING CLASSES
        # ====================================================

        classes = np.unique(
            y_train
        )


        for class_value in classes:

            points = X_visual[
                y_train == class_value
            ]


            ax.scatter(

                points[:, 0],

                points[:, 1],

                s=70,

                label=(
                    "Class "
                    +
                    str(
                        label_encoder.classes_[
                            class_value
                        ]
                    )
                )
            )


        # ====================================================
        # 43. PLOT TEST POINT
        # ====================================================

        ax.scatter(

            test_point[0],

            test_point[1],

            marker="*",

            s=350,

            edgecolors="black",

            linewidth=2,

            label="Test Point"
        )


        # ====================================================
        # 44. AXIS LABELS
        # ====================================================

        ax.set_xlabel(

            feature_columns[0],

            fontsize=12
        )


        ax.set_ylabel(

            feature_columns[1],

            fontsize=12
        )


        # ====================================================
        # 45. TITLE
        # ====================================================

        ax.set_title(

            "KNN Classification - Animated",

            fontsize=16
        )


        # ====================================================
        # 46. GRID
        # ====================================================

        ax.grid(

            True,

            alpha=0.3
        )


        # ====================================================
        # 47. LEGEND
        # ====================================================

        ax.legend()


        # ====================================================
        # 48. INFORMATION BOX
        # ====================================================

        info = ax.text(

            0.02,

            0.97,

            "",

            transform=ax.transAxes,

            verticalalignment="top",

            fontsize=10,

            bbox=dict(

                boxstyle="round",

                facecolor="white",

                alpha=0.9
            )
        )


        # ====================================================
        # 49. STORE ANIMATION LINES
        # ====================================================

        lines = []


        # ====================================================
        # 50. ANIMATION UPDATE FUNCTION
        # ====================================================

        def update(frame):


            # ------------------------------------------------
            # Remove previous lines
            # ------------------------------------------------

            for line in lines:

                line.remove()


            lines.clear()


            # =================================================
            # STEP 1
            # =================================================

            if frame == 0:

                info.set_text(

                    "STEP 1\n\n"

                    "Test point selected\n"

                    "Waiting to calculate distances..."
                )


            # =================================================
            # STEP 2
            # DISTANCE CALCULATION
            # =================================================

            elif 1 <= frame <= len(
                animation_indices
            ):

                index = animation_indices[
                    frame - 1
                ]


                point = X_visual[
                    index
                ]


                line, = ax.plot(

                    [

                        test_point[0],

                        point[0]
                    ],

                    [

                        test_point[1],

                        point[1]
                    ],

                    linestyle="--",

                    linewidth=1
                )


                lines.append(line)


                info.set_text(

                    "STEP 2: DISTANCE CALCULATION\n\n"

                    f"Training Point : {index + 1}\n"

                    f"Distance       : "
                    f"{distances[index]:.3f}"
                )


            # =================================================
            # STEP 3
            # K NEAREST NEIGHBOURS
            # =================================================

            elif frame == len(
                animation_indices
            ) + 1:


                for index in nearest_indices:

                    point = X_visual[
                        index
                    ]


                    line, = ax.plot(

                        [

                            test_point[0],

                            point[0]
                        ],

                        [

                            test_point[1],

                            point[1]
                        ],

                        linewidth=3
                    )


                    lines.append(line)


                neighbour_classes = y_train[
                    nearest_indices
                ]


                neighbour_names = [

                    label_encoder.classes_[c]

                    for c in neighbour_classes
                ]


                info.set_text(

                    "STEP 3: K NEAREST NEIGHBOURS\n\n"

                    f"K = {K}\n"

                    f"Nearest Points = "
                    f"{nearest_indices + 1}\n"

                    f"Classes = "
                    f"{neighbour_names}"
                )


            # =================================================
            # STEP 4
            # MAJORITY VOTING
            # =================================================

            elif frame == len(
                animation_indices
            ) + 2:


                for index in nearest_indices:

                    point = X_visual[
                        index
                    ]


                    line, = ax.plot(

                        [

                            test_point[0],

                            point[0]
                        ],

                        [

                            test_point[1],

                            point[1]
                        ],

                        linewidth=3
                    )


                    lines.append(line)


                neighbour_classes = y_train[
                    nearest_indices
                ]


                unique, counts = np.unique(

                    neighbour_classes,

                    return_counts=True
                )


                voting_text = ""


                for c, count in zip(

                    unique,

                    counts
                ):

                    voting_text += (

                        f"Class "
                        f"{label_encoder.classes_[c]}"

                        f" → {count} vote(s)\n"
                    )


                info.set_text(

                    "STEP 4: MAJORITY VOTING\n\n"

                    +
                    voting_text
                )


            # =================================================
            # STEP 5
            # FINAL PREDICTION
            # =================================================

            else:


                for index in nearest_indices:

                    point = X_visual[
                        index
                    ]


                    line, = ax.plot(

                        [

                            test_point[0],

                            point[0]
                        ],

                        [

                            test_point[1],

                            point[1]
                        ],

                        linewidth=3
                    )


                    lines.append(line)


                info.set_text(

                    "STEP 5: FINAL PREDICTION\n\n"

                    f"K = {K}\n"

                    f"Actual Class    : "

                    f"{label_encoder.classes_[actual_class]}\n"

                    f"Predicted Class : "

                    f"{label_encoder.classes_[predicted_class]}"
                )


            return lines + [info]


        # ====================================================
        # 51. CREATE ANIMATION
        # ====================================================

        total_frames = (

            len(animation_indices)

            + 4
        )


        anim = FuncAnimation(

            fig,

            update,

            frames=total_frames,

            interval=350,

            repeat=True,

            blit=False
        )


        # ====================================================
        # 52. DISPLAY ANIMATION
        # ====================================================

        plt.close(fig)


        animation_html = anim.to_jshtml()


        display(

            HTML(
                animation_html
            )
        )


# ============================================================
# 53. CONNECT BUTTON
# ============================================================

run_button.on_click(
    run_knn
)


# ============================================================
# 54. READY MESSAGE
# ============================================================

print("\n" + "=" * 60)
print("READY!")
print("=" * 60)

print(
    "\n1. Select the X-axis feature."
)

print(
    "2. Select the Y-axis feature."
)

print(
    "3. Select the target column."
)

print(
    "4. Choose K."
)

print(
    "5. Click 'Run KNN'."
)

                  KNN CLASSIFICATION

Upload your dataset.
Supported formats: CSV, XLSX, XLS


Saving healthcare-dataset-stroke-data.csv to healthcare-dataset-stroke-data (2).csv

Uploaded file: healthcare-dataset-stroke-data (2).csv

DATASET LOADED SUCCESSFULLY

Number of Rows    : 5110
Number of Columns : 12

Column Names:
1. id
2. gender
3. age
4. hypertension
5. heart_disease
6. ever_married
7. work_type
8. Residence_type
9. avg_glucose_level
10. bmi
11. smoking_status
12. stroke

First 5 Rows:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1



NUMERICAL FEATURES
• id
• age
• hypertension
• heart_disease
• avg_glucose_level
• bmi
• stroke

SELECT TWO FEATURES

Your dataset contains 7 numerical features.
Select exactly TWO different numerical features for the KNN visualization.


Dropdown(description='X-axis:', layout=Layout(width='400px'), options=('id', 'age', 'hypertension', 'heart_dis…

...


SELECT VALUE OF K


IntSlider(value=3, description='K:', layout=Layout(width='400px'), max=15, min=1, style=SliderStyle(descriptio…

Button(button_style='success', description='Run KNN', icon='play', layout=Layout(height='40px', width='200px')…

Output()


READY!

1. Select the X-axis feature.
2. Select the Y-axis feature.
3. Select the target column.
4. Choose K.
5. Click 'Run KNN'.
